In [1]:
from pytential import sympy_pytential, min_pytential, quad_pytential
import numpy as np
from sympy import log, symbols
import plotly.graph_objects as go

This file demonstrates how to create, combine, and manipulate pytentials.  Equibilriation is used to reduce free variables. 

# Create phases

Create two binary ideal solutions and add the lattice constraint in terms of the phase volume. 

In [2]:
# Assemble sympy expressions.
c0a, c1a, c0b, c1b, Va, Vb = symbols('c0a, c1a, c0b, c1b, Va, Vb')

RT = 8.134*300
fa_sp = c0a*RT*(2+log(c0a/(c0a+c1a))) +c1a*RT*(0+log(c1a/(c0a+c1a)))
fb_sp = c0b*RT*(0+log(c0b/(c0b+c1b))) +c1b*RT*(1+log(c1b/(c0b+c1b)))
lattice_constraint_a = c0a + c1a - Va
lattice_constraint_b = c0b + c1b - Vb

In [3]:
# Build the pytential with constraint. 
fa = sympy_pytential(fa_sp, constraints_sym=[lattice_constraint_a])
fb = sympy_pytential(fb_sp, constraints_sym=[lattice_constraint_b])

In [4]:
x_values = np.linspace(0.01, .99, 100)
Fa = fa(c0a=x_values, c1a=1-x_values, Va=1)
fa_trace = go.Scatter(x=x_values, y=Fa, mode='lines', name='fa')

Fb = fb(c0b=x_values, c1b=1-x_values, Vb=1)
fb_trace = go.Scatter(x=x_values, y=Fb, mode='lines', name='fb')

fig_fafb = go.Figure(data = [fa_trace, fb_trace])
fig_fafb.update_layout(
    xaxis_title='c0',
    yaxis_title='Energy',
)
fig_fafb.show()

# Combine functions

We now combine both functions into a composite pytential, and add constraints for the total of each species. 

Note the pytential takes c0 and c1 as arguments even though they only appear in the constraints. 

In [5]:
f = fa+fb
# Define and add constraints
c0, c1 = symbols('c0, c1')
f = f.add_constraints_sym([c0a+c0b-c0, c1a+c1b-c1]) 
print(f)

x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 2) + 2440.2*c0b*log(c0b/(c0b + c1b)) + 2440.2*c1a*log(c1a/(c0a + c1a)) + 2440.2*c1b*(log(c1b/(c0b + c1b)) + 1)

f'(x)= [0, 0, 0, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 2440.2*log(c0a/(c0a + c1a)) + 4880.4, -2440.2*c1b/(c0b + c1b) + 2440.2*(c0b + c1b)*(-c0b/(c0b + c1b)**2 + 1/(c0b + c1b)) + 2440.2*log(c0b/(c0b + c1b)), 0, -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 2440.2*log(c1a/(c0a + c1a)), -2440.2*c0b/(c0b + c1b) + 2440.2*(c0b + c1b)*(-c1b/(c0b + c1b)**2 + 1/(c0b + c1b)) + 2440.2*log(c1b/(c0b + c1b)) + 2440.2]

f"(x)= [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, -2440.2*c0a/(c0a + c1a)**2 + 2440.2*c1a/(c0a + c1a)**2 + 2440.2*(c0a + c1a)*(2*c0a/(c0a + c1a)**3 - 2/(c0a + c1a)**2) + 2440.2/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a))

# Equilibrium states

## Zero pressure

Given a particular composition, we want to find the equilibrium (minimum) energy of the total system. To do this, we define a new pytential from $f$ which is a function of the overall composition and determine the remaining arguments by minimization. 

In [6]:
f_min = min_pytential(f.add_constraints_sym([Va+Vb-1]), ['c0'])
print(f_min)

cont []
Pytential of type <class 'pytential.min_pytential.min_pytential.min_pytential'>
Variables: ['c0']
Potential: <bound method args_to_list.<locals>.wrapper of <pytential.min_pytential.min_pytential.min_pytential object at 0x0000025797BC08F0>>
Gradient: <bound method args_to_list.<locals>.wrapper of <pytential.min_pytential.min_pytential.min_pytential object at 0x0000025797BC08F0>>
Hessian: <bound method args_to_list.<locals>.wrapper of <pytential.min_pytential.min_pytential.min_pytential object at 0x0000025797BC08F0>>



Note that since the function $f^{min}$ is no longer a sympy expression, it's function is reference to a bound method (the minimizer).

Let's use the minimizer to see the equilbrium partitioning. 

In [7]:
ym = f_min(x_values)
f_min_trace = go.Scatter(x=x_values, y=ym, mode='lines', name='f_min')
fig_fafbfmin = fig_fafb
fig_fafbfmin.add_trace(f_min_trace)
fig_fafbfmin.show()

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:441: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:495: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:441: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\opti

The minimal energy partitions the overall composition, $c_0$, between phases subject to $c_0=c_0^a+c_0^b$, and similarly for $c_1$. Graphically, this minimum is the lowest common tangent between $f^a$ and $f^b$. Along this path, the volume the volume of $a$ decreases $V^a: 1->0$ and $V^b: 0->1$.

At any point along the tangent, we can check the equilibrium partitioning:

In [8]:
y0 = f_min.min_fcn(np.array([[.4]]))[1][0]
print("An equilibrium partitioning is :", y0)
print('f ', f(**y0), ', fa ', fa(**y0), ', fb ', fb(**y0), ', fa + fb', fa(**y0) + fb(**y0))

An equilibrium partitioning is : {'c0': 0.4, 'Va': 0.46111985011250894, 'Vb': 0.5388801498874911, 'c0a': 0.04151450068326094, 'c0b': 0.3584854993167391, 'c1': 0.6, 'c1a': 0.419605349429248, 'c1b': 0.18039465057075202}
f  -535.9873725019713 , fa  -137.89306993617276 , fb  -398.09430256579856 , fa + fb -535.9873725019713


c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds



## Controlling V

Consider adding the volume $V$ as another dimension to this surface. We can then visualize it along with the original phases and the lowest common tangent.

In [9]:
f_min2 = min_pytential(f.add_constraints_sym([Va+Vb-1, c0+c1-1]), ['c0', 'Va'])
print(f_min2.min_fcn(np.array([.7,.5])))

(0.0005549289999904858, {'c0': 0.7, 'Va': 0.5, 'Vb': 1e-06, 'c0a': 1e-06, 'c0b': 1e-06, 'c1': 1e-06, 'c1a': 1e-06, 'c1b': 1e-06})


In [10]:
# The minimizer can be slow, so we define a reduced set of coordinates
x_values2 = np.linspace(0.01, .99, 10)
X, Y = np.meshgrid(x_values2, x_values2)
F_min2 = f_min2(c0=X.ravel(), Va=Y.ravel()).reshape(X.shape)

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:441: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:495: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds



In [11]:
# Create a Plotly surface plot
fig = go.Figure()

fig.add_trace(go.Surface(z=F_min2, x=x_values2, y=x_values2, colorscale='Viridis'))

# Add a line plot for f_a at Va = 1
fig.add_trace(go.Scatter3d(
    x=x_values,
    y=1+0*x_values,  # Va = 1
    z=Fa,
    mode='lines',
    name='f_a',
    line=dict(color='blue', width=6)
))

# Add a line plot for f_b at Va = 0
fig.add_trace(go.Scatter3d(
    x=x_values,
    y=0*x_values,  # Va = 0
    z=Fb,
    mode='lines',
    name='f_b',
    line=dict(color='green', width=6)
))

# Add labels and title
fig.update_layout(
    scene=dict(
        xaxis_title="c0", yaxis_title="Va", zaxis_title="Energy",
    ),
)

# Show the plot
fig.show()

In [12]:
fq = quad_pytential.from_homog_pyt(f.add_constraints_sym([Va+Vb-1, c0+c1-1]), y0)
print(fq)
print(fq(**y0))
print(f(**y0))


x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 0.5*c0a*(53487.561922346*c0a - 5291.89970764567*c1a) - 994.662629341261*c0a + 0.5*c0b*(2278.69027108106*c0b - 4528.27961933553*c1b) - 994.637669142093*c0b + 0.5*c1a*(-5291.89970764567*c0a + 523.564759905065*c1a) - 230.216673028996*c1a + 0.5*c1b*(-4528.27961933553*c0b + 8998.72903795798*c1b) - 230.22368442056*c1b

f'(x)= [0, 0, 0, 53487.561922346*c0a - 5291.89970764567*c1a - 994.662629341261, 2278.69027108106*c0b - 4528.27961933553*c1b - 994.637669142093, 0, -5291.89970764567*c0a + 523.564759905065*c1a - 230.216673028996, -4528.27961933553*c0b + 8998.72903795798*c1b - 230.22368442056]

f"(x)= [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 53487.5619223460, 0, 0, -5291.89970764567, 0], [0, 0, 0, 0, 2278.69027108106, 0, 0, -4528.27961933553], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, -5291.89970764567, 0, 0, 523.564759905065, 0], [0, 0, 0, 0, -4528.27961933553, 0, 0, 8998.72903795798]]

Co

In [13]:
fq2 = fq.reduce_by_eliminating_linear_constraints(vars_to_keep = ['c0', 'Va'])


['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']
['c0', 'Va']
[2, 0]


This did not eliminate all the dependent variables since the constraints were linearly dependent. We can still obtain an answer but now we need to minimize over c0b. 

In [14]:
fq3 = fq2.reduce_uncontrained_by_minimization(vars_to_keep = ['c0', 'Va'])
print(fq2)
print(fq3)

[0, 1]
x = ['c0', 'Va', 'c1b']

f(x) = 0.5*Va*(55766.2521934271*Va + 58779.4616299917*c0 + 65586.4315204083*c1b) - 61058.1768612719*Va + 0.5*c0*(58779.4616299917*Va + 64594.9260975425*c0 + 64594.9260975425*c1b) - 65359.3720538547*c0 + 0.5*c1b*(65586.4315204083*Va + 64594.9260975425*c0 + 84928.9046452526*c1b) - 71401.9279595498*c1b + 33206.6164714819

f'(x)= [58779.4616299917*Va + 64594.9260975425*c0 + 64594.9260975425*c1b - 65359.3720538547, 55766.2521934271*Va + 58779.4616299917*c0 + 65586.4315204083*c1b - 61058.1768612719, 65586.4315204083*Va + 64594.9260975425*c0 + 84928.9046452526*c1b - 71401.9279595498]

f"(x)= [[64594.9260975425, 58779.4616299917, 64594.9260975425], [58779.4616299917, 55766.2521934271, 65586.4315204083], [64594.9260975425, 65586.4315204083, 84928.9046452526]]
x = ['c0', 'Va']

f(x) = 0.5*Va*(5117.06488141739*Va + 8895.96537205283*c0) - 5917.96661006256*Va + 0.5*c0*(8895.96537205284*Va + 15465.5455294603*c0) - 11052.746070224*c0 + 3191.89216980848

f'(x)= [8895.96

In [15]:
import copy

fig_q = copy.deepcopy(fig_fafbfmin)
fig_q.add_trace(go.Scatter(x=x_values, y=fq3(c0=x_values, Va=0), mode='lines', name='fq3'))
fig_q.add_trace(go.Scatter(x=x_values, y=fq3(c0=x_values, Va=1), mode='lines', name='fq3'))

fig_q.show()

Fq3 = fq3(c0=X.ravel(),  Va=Y.ravel()).reshape(X.shape)

In [16]:
fq3_trace = go.Surface(z=Fq3, x=x_values2, y=x_values2, colorscale='Viridis', name='fq2')
fig = go.Figure(data=[fq3_trace])
# Add a line plot for f_a at Va = 1
fig.add_trace(go.Scatter3d(
    x=x_values,
    y=1+0*x_values,  # Va = 1
    z=Fa,
    mode='lines',
    name='f_a',
    line=dict(color='blue', width=6)
))

# Add a line +
# plot for f_b at Va = 0
fig.add_trace(go.Scatter3d(
    x=x_values,
    y=0*x_values,  # Va = 0
    z=Fb,
    mode='lines',
    name='f_b',
    line=dict(color='green', width=6)
))
fig.show()

In [17]:
fqc = quad_pytential.from_homog_pyt(f.add_constraints_sym([Va+Vb-1]), y0)
fqc2 = fqc.reduce_by_eliminating_linear_constraints(vars_to_keep = ['c0', 'c1', 'Va'])
fqc3 = fqc2.reduce_uncontrained_by_minimization(vars_to_keep = ['c0', 'c1', 'Va'])
Fqc3 = fqc3(c0=X.ravel(), c1=Y.ravel(), Va=0.5).reshape(X.shape)

['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']
['c0', 'c1', 'Va']
[2, 5, 0]
[0, 1, 2]


In [18]:
fqc3_trace = go.Surface(z=Fqc3, x=x_values2, y=x_values2, colorscale='Viridis', name='fqc3')
fig = go.Figure(data=[fqc3_trace])
fig.show()